# 301 · Backtesting & Evaluation

Before deploying a new sequential design (like GSD or AVI) to production, you should verify its performance on historical data. This process, called **Backtesting**, allows you to answer questions like:
- "How much earlier would we have stopped on this past experiment?"
- "Would we have made a different decision?"

In this tutorial, we will:
- Load the **ASOS Online-Controlled-Experiment dataset**.
- Run a historical simulation using the unified `backtest()` API.

## 1. Data Preparation

We use the open ASOS digital experiments dataset. We'll pick one experiment and extract incremental batches.

In [ ]:
from pathlib import Path

import ibis
import pandas as pd

from earlysign.core.ledger import Ledger
from earlysign.v1.controllers.GST_Spending_JennisonTurnbull2000 import (
    JennisonTurnbull2000Controller,
)

# Download dataset if needed
data_path = Path("data/asos_digital_experiments_dataset.parquet")
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    !wget -O {data_path} https://osf.io/62t7f/download

df_raw = pd.read_parquet(data_path)
exp_id = "3b4300"

# NOTE: For backtesting, we must filter for a specific test group (variant_id)
# because the dataset may contain multiple treatments for the same experiment.
df = df_raw[
    (df_raw["experiment_id"] == exp_id)
    & (df_raw["metric_id"] == 1)
    & (df_raw["variant_id"] == 1)
].copy()

print(f"Loaded {len(df)} monitoring steps for experiment {exp_id} (variant 1).")

## 2. Preparing a Backtest Table

The `backtest_from_table()` API expects an Ibis table with columns for arm names, sample sizes, and successes.

In [ ]:
df = df.sort_values("time_since_start")
# Calculate increments between monitoring steps
df["dn_c"] = df["count_c"].diff().fillna(df["count_c"]).astype(int)
df["ds_c"] = (
    (df["count_c"] * df["mean_c"]).diff().fillna(df["count_c"] * df["mean_c"])
).astype(int)
df["dn_t"] = df["count_t"].diff().fillna(df["count_t"]).astype(int)
df["ds_t"] = (
    (df["count_t"] * df["mean_t"]).diff().fillna(df["count_t"] * df["mean_t"])
).astype(int)

# Convert to long-format table for Ibis
df_long = pd.concat(
    [
        df[["time_since_start", "dn_c", "ds_c"]]
        .rename(columns={"dn_c": "total", "ds_c": "success"})
        .assign(arm="control"),
        df[["time_since_start", "dn_t", "ds_t"]]
        .rename(columns={"dn_t": "total", "ds_t": "success"})
        .assign(arm="treatment"),
    ]
).sort_values("time_since_start")

tbl_events = ibis.memtable(df_long)
tbl_events.head().execute()

## 3. Running the Backtest

The `backtest_from_table()` API expects an Ibis table with columns for arm names, sample sizes, and successes.
We initialize a design (GSD) and run the backtest. The controller will process the table as if it were arriving in real-time.

In [ ]:
ledger = Ledger(ibis.connect("duckdb://:memory:"), "backtest")
ledger.ensure()
trial = JennisonTurnbull2000Controller(ledger)

# Design a protocol based on baseline rates from the data
p_baseline = df.iloc[0]["mean_c"]
protocol = trial.design(
    p_control=p_baseline,
    effect_spec={"kind": "absolute_difference", "value": 0.005},
    alpha=0.05,
    power=0.8,
    looks=3,
)
trial.set_protocol(protocol)

print("Starting backtest...")
results = trial.backtest_from_table(
    tbl_events,
    arm_col="arm",
    total_col="total",
    success_col="success",
    order_by="time_since_start",
)

print(f"Backtest Final Status: {results['final_status']}")
print(f"Sample Size at Stop: {results['sample_n']:,}")

## 4. Visualizing Backtest Results

Just as we did in the GSD tutorial, we can visualize the operating characteristics of the design and the trajectory of the test statistic during the backtest.

In [ ]:
from earlysign.v1.methods.group_sequential.plan.visualize import (
    visualize_protocol_design,
)

# 1. Visualize the design (OC)
res = visualize_protocol_design(protocol, effect_sizes=list(range(-20, 50 + 5, 5)))
display(res["summary"])
display(res["figure"])

# 2. Visualize the trajectory of the backtest
trial.plot_result()

## 5. Summary

- **Evaluation**: Backtesting provides empirical evidence that a design is suitable for your specific data distribution.
- **Comparison**: You can run multiple backtests with different controllers (GSD vs mSPRT) to see which would have performed better.
- **Continuity**: The same code used for backtesting can be used for live orchestration.

In the final tutorial, we will learn how to deploy these designs to **BigQuery** for production usage.